# Lookback/Lookahead FL Create & Store (66-wide peak variant)

Builds the feature/label matrix for an asset using
`asset_snapshot_lookback_lookahead_normalize_top_prominences_prepare_all`,
which appends 6 KDE top-prominence peak features (indices 60-65) to the
original 60-element vector, and stores the result to GCS as
`fl_data_peak_{ASSET}` (shape `(n_obs, 66)`).

> Note: per-observation scipy peak-finding makes `_all` much slower than the
> njit/prange 60-wide original (~1M obs for btcusdt). Test on a small slice of
> `candles` first if needed.

In [ ]:
%pip install -q numpy numba scipy google-cloud-storage google-auth

In [ ]:
import os
import sys

REPO_URL = "https://github.com/payamdav/pycrypto.git"
REPO_NAME = "pycrypto"
BRANCH_NAME = "claude/gallant-ptolemy-b5i7km"  # branch this notebook lives on

if not os.path.exists(REPO_NAME):
    !git clone -b {BRANCH_NAME} {REPO_URL}

REPO_PATH = os.path.abspath(REPO_NAME)
if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)

sys.path.insert(0, os.path.join(REPO_PATH, "packages/tools/google_cloud_storage_tools"))

import time
import io
import numpy as np
from gcs_tools import read_file, write_file, gcs_json_key_file

gcs_json_key_file()

In [ ]:
ASSET      = "btcusdt"
# ASSET     = "ethusdt"
# ASSET     = "adausdt"
# ASSET     = "xrpusdt"
# ASSET     = "trumpusdt"
# ASSET     = "vineusdt"
# ASSET     = "dogeusdt"

GCS_BUCKET = "payamdprojectbucket"
GCS_KEY    = f"lookback_lookahead_{ASSET}"

t0 = time.perf_counter()
data_bytes = read_file(GCS_BUCKET, GCS_KEY)
elapsed = time.perf_counter() - t0

candles = np.load(io.BytesIO(data_bytes))

file_size = len(data_bytes)
print(f"Loaded from gs://{GCS_BUCKET}/{GCS_KEY}")
print(f"File size : {file_size / 1024 / 1024:.2f} MB  ({file_size:,} bytes)")
print(f"Load time : {elapsed:.4f}s")
print(f"Array shape: {candles.shape}  dtype: {candles.dtype}")

In [ ]:
LOOK_BACK  = 1440
LOOK_AHEAD = 240
K          = 100

# KDE parameters (defaults from look_back_look_ahead.ipynb cell 1)
KDE_BINS           = 200
KDE_BANDWIDTH      = 5
KDE_KERNEL         = "Triangular"
KDE_IGNORE_BORDERS = True
KDE_PEAK_DISTANCE  = 5

In [ ]:
from packages.asset_analyzer import asset_snapshot_lookback_lookahead_normalize_top_prominences_prepare_all

# Heads-up: per-observation scipy peak-finding makes this slow over the full
# btcusdt array (~1M obs). To smoke-test first, slice candles, e.g.:
#   candles_small = candles[:LOOK_BACK + LOOK_AHEAD + 1000]
#   fl = asset_snapshot_lookback_lookahead_normalize_top_prominences_prepare_all(
#       candles_small, LOOK_BACK, LOOK_AHEAD, K,
#       kde_bins=KDE_BINS, kde_bandwidth=KDE_BANDWIDTH, kde_kernel=KDE_KERNEL,
#       kde_ignore_borders=KDE_IGNORE_BORDERS, kde_peak_distance=KDE_PEAK_DISTANCE)

t0 = time.perf_counter()
fl = asset_snapshot_lookback_lookahead_normalize_top_prominences_prepare_all(
    candles, LOOK_BACK, LOOK_AHEAD, K,
    kde_bins=KDE_BINS,
    kde_bandwidth=KDE_BANDWIDTH,
    kde_kernel=KDE_KERNEL,
    kde_ignore_borders=KDE_IGNORE_BORDERS,
    kde_peak_distance=KDE_PEAK_DISTANCE,
)
elapsed = time.perf_counter() - t0
print(f"prepare_all elapsed: {elapsed:.4f}s  shape: {fl.shape}  size: {fl.nbytes / 1024 / 1024:.2f} MB")
assert fl.shape[1] == 66, fl.shape

In [ ]:
GCS_BUCKET = "payamdprojectbucket"
GCS_KEY    = f"fl_data_peak_{ASSET}"

buf = io.BytesIO()
np.save(buf, fl)
buf.seek(0)

t0 = time.perf_counter()
write_file(GCS_BUCKET, GCS_KEY, buf)
elapsed = time.perf_counter() - t0
print(f"Written to gs://{GCS_BUCKET}/{GCS_KEY}  elapsed: {elapsed:.4f}s")

In [ ]:
%pip install -q matplotlib

import matplotlib.pyplot as plt

# 6 new KDE peak features (indices 60-65)
peak_labels = [
    "above_peak_1 (60)", "above_peak_2 (61)", "above_peak_3 (62)",
    "below_peak_1 (63)", "below_peak_2 (64)", "below_peak_3 (65)",
]
colors = ["#55A868", "#4C72B0", "#64B5CD", "#C44E52", "#8172B2", "#CCB974"]
bins = np.arange(-1.0, 1.05, 0.05)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle("KDE Top-Prominence Peak Features (normalized vwap positions)", fontsize=14, fontweight="bold")
for j, ax in enumerate(axes.ravel()):
    col = fl[:, 60 + j]
    ax.hist(col, bins=bins, color=colors[j], edgecolor="white", linewidth=0.4)
    ax.set_title(peak_labels[j], fontsize=11, fontweight="bold")
    ax.set_xlabel("Normalized vwap position")
    ax.set_ylabel("Count")
    ax.set_xlim(-1.05, 1.05)
    ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()